# Histogram Analysis

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

pd.set_option("display.max_columns", None)

In [ ]:
from utils.constants import RANDOM_SEED
from utils.common import get_data_folder_path, set_plotting_config

In [ ]:
# plots configuration
sns.set_style("darkgrid")
sns.set_palette("colorblind")
set_plotting_config()
%matplotlib inline

## 1. Preprocessing

### Load data

In this notebook, we will use the **Medical Insurance Payout Dataset**. This dataset contains historical data for over 1300 insurance customers (age, sex, BMI, number of children, smoking habits, and region) along with their actual medical charges. i.e., the expenditure for the customer.

Sources:
1. Kaggle: https://www.kaggle.com/datasets/harshsingh2209/medical-insurance-payout
2. Original source: https://raw.githubusercontent.com/JovianML/opendatasets/master/data/medical-charges.csv

In [ ]:
data_path = get_data_folder_path()

df_input = pd.read_csv(os.path.join(data_path, "expenses.csv"))

In [ ]:
from typing import Any

In [ ]:
from matplotlib import ticker as mticker

In [ ]:
def _build_histogram(
    ax: plt.Axes,
    df: pd.DataFrame,
    plot_col: str,
    display_name: str = None,
    display_unit: str = None,
    stratify_col: str = None,
    bin_size: int = None,
    linewidth: int = 1.5,
    histogram_opacity: float = 0.75,
    show_percentage: bool = False,
    show_mean: bool = True,
    show_median: bool = False,
    show_zero_line: bool = False,
    show_kde: bool = True,
) -> plt.Axes:
    
    display_name = display_name or plot_col

    ax = sns.histplot(
        df,
        x=plot_col,
        kde=show_kde,
        ax=ax,
        bins=np.arange(
            (df[plot_col].min() // bin_size) * bin_size,
            (df[plot_col].max() + bin_size),
            bin_size
        ),
        alpha=histogram_opacity,
        line_kws=dict(linewidth=linewidth),
        label=None if stratify_col else display_name,
        stat="percent" if show_percentage else "count",
        hue=stratify_col,
        multiple="stack" if stratify_col else "layer",
        zorder=1
    )
        
    if show_percentage:
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
        ax.set_ylabel("Percentage")
    else:
        ax.set_ylabel("Count")
    # add thousands separator to x-axis
    ax.xaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
    ax.set_xlabel(display_name + (f" ({display_unit})" if display_unit else ""))
    
    color_lst = sns.color_palette()
    color_idx = df[stratify_col].nunique() if stratify_col else 1
    s = df[plot_col]
    if show_mean is True:
        # Plot vertical line for mean
        mean = s.mean()
        label_mean = f"Mean ({mean:,.0f}" + (f" {display_unit})" if display_unit else ")")
        ax.axvline(mean, linewidth=linewidth, color=color_lst[color_idx], label=label_mean, zorder=3)
    if show_median is True:
        # Plot dotted vertical line for median
        median = s.median()
        label_median = f"Median ({median:,.0f}" + (f" {display_unit})" if display_unit else ")")
        ax.axvline(median, linestyle="--", linewidth=linewidth, color=color_lst[color_idx], label=label_median, zorder=4)
        
    if show_zero_line is True:
        # Plot vertical line in zero
        ax.axvline(0, linestyle="-.", linewidth=linewidth/2, color="black", label=None, zorder=2)
    
    # configure legend
    legend = ax.get_legend()
    handles, labels = ax.get_legend_handles_labels()
    if legend:
        ax.legend(
            handles=list(legend.get_patches()) + handles,
            labels=[f"{display_name} ({stratify_col.title()}: {txt.get_text().title()})" for txt in legend.get_texts()] + labels,
            title=None
        )
    else:
        # bring last legend item (histogram) to the front
        bring_last_to_front = lambda lst: [lst[-1], *lst[:-1]]
        ax.legend(
            handles=bring_last_to_front(handles),
            labels=bring_last_to_front(labels),
            title=None
        )
    
    return ax
    
    

In [ ]:
with sns.axes_style("darkgrid"):

    # Create a figure and a set of subplots
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))

    ax = _build_histogram(
        df=df_input,
        plot_col="charges",
        display_name="Medical Charges",
        display_unit="USD",
        ax=ax,
        # stratify_col="smoker",
        bin_size=2000,
        show_percentage=True,
        linewidth=1.5,
        show_mean=True,
        show_median=True,
        show_zero_line=False,
        show_kde=False,
    )
    
    

In [ ]:
def plot_comparison_histograms(maint_type: str, duration_col: str, comparison_dfs: dict[str, pd.DataFrame], contract_to_filter: str = None, show_median: bool = False):
    
    df_comparison = comparison_dfs[maint_type].copy()
    
    if contract_to_filter is not None:
        df_comparison = df_comparison[df_comparison["contract_type"] == contract_to_filter]
    
    title = f"Manutenção {maint_type.capitalize()} (n = {len(df_comparison):,}) [{duration_case}]"
    if contract_to_filter is not None:
        title += f" - Contrato "{contract_to_filter}""

    with sns.axes_style("darkgrid"):

        # Create a figure and a set of subplots
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
        fig.suptitle(title, fontsize=16)
        
        # LEFT PLOT
        plot_histogram(
            df=df_comparison,
            plot_col=f"{duration_col}_before",
            ax=ax1,
            color="red",
            label_histogram="Durations before PM",
            label_lines="duration before",
            bins=np.arange(0, df_comparison[f"{duration_col}_before"].max(), 100),
            show_median=show_median,
        )
        plot_histogram(
            df=df_comparison,
            plot_col=f"{duration_col}_after",
            ax=ax1,
            color="blue",
            label_histogram="Durations after PM",
            label_lines="duration after",
            bins=np.arange(0, df_comparison[f"{duration_col}_after"].max(), 100),
            show_median=show_median,
        )
        # title and labels
        ax1.set_title("Periods Before and After Preventive Maintenance (PM)", fontsize=14)
        ax1.set_xlabel("Period duration (hrs)")
        ax1.set_ylabel("Frequency")
        # x axis range
        ax1.set_xlim([0, 4000])
        # legend
        handles, labels = ax1.get_legend_handles_labels()
        order = [4, 5, 1, 3, 0, 2] if show_median is True else [2, 3, 0, 1]
        ax1.legend([handles[idx] for idx in order],[labels[idx] for idx in order])
        
        # RIGHT PLOT
        plot_histogram(
            df=df_comparison,
            plot_col=f"{duration_col}_diff",
            ax=ax2,
            color="green",
            label_histogram="Duration difference",
            label_lines="difference",
            bins=np.arange(
                (df_comparison[f"{duration_col}_diff"].min() // 100) * 100,
                (df_comparison[f"{duration_col}_diff"].max() + 100),
                100
            ),
            show_zero_line=True,
            show_median=show_median,
        )
        # title and labels
        ax2.set_title("Difference between Periods right Before and After PM", fontsize=14)
        ax1.set_xlabel("Period duration (hrs)")
        ax1.set_ylabel("Frequency")
        # x axis range
        ax2.set_xlim([-4000, 4000])
        # legend
        handles, labels = ax2.get_legend_handles_labels()
        order = [2, 1, 0] if show_median is True else [1, 0]
        ax2.legend([handles[idx] for idx in order],[labels[idx] for idx in order])
        
        plt.tight_layout()
        
        return fig
    
